# Part 3 Starter Notebook

This notebook loads an AIME 2024 dataset, runs a model on each problem, extracts an AIME-style final answer, and grades the outputs.

In [7]:
import re

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [10]:
import getpass
import os

token = getpass.getpass("Enter HF_TOKEN: ")
os.environ["HF_TOKEN"] = token

In [11]:
print("Token set" if os.environ.get("HF_TOKEN") else "Token not set")

Token set


In [12]:
# MODEL_NAME = "Qwen/Qwen3-4B"
# or allenai/Olmo-3-7B-Think
MODEL_NAME = "allenai/Olmo-3-7B-Think"
DATASET_NAME = "OpenRLHF/aime-2024"
MAX_NEW_TOKENS = 512

## Loading the model and the data

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"

hf_token = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    token=hf_token,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
dataset = load_dataset(DATASET_NAME, split="train", token=hf_token)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

## Evaluation helpers

In [ ]:
def strip_thinking_trace(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|begin_of_thought\|>.*?<\|end_of_thought\|>", "", text, flags=re.DOTALL)
    return text.strip()



def extract_answer(text: str, mode="exact_match") -> int | None:
    """Extract an AIME-style integer answer from a model completion."""
    answer_text = strip_thinking_trace(text)
    if not answer_text:
        if mode == "exact_match":
            return None
        else:
            answer_text = text  # fall back to full text

    
    # 1. Boxed LaTeX answer: \boxed{123}
    if mode == "exact_match":
        boxed = re.findall(r"\\boxed\{(\d+)\}", answer_text)
        if boxed:
            val = int(boxed[-1])
            return val
        else:
            return None
        
    elif mode == "flexible_extract":
        # 2. "The answer is N" or "answer: N" patterns
        patterns = [
            r"(?:the\s+)?answer\s+is\s+[:\s]*(\d+)",
            r"answer[:\s]+(\d+)",
            r"=\s*(\d+)\s*$",
            r"(?:therefore|thus|so),?\s+(\d+)\s*(?:\.|$)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, answer_text, re.IGNORECASE)
            if matches:
                val = int(matches[-1])
                return val

        # 3. Last integer in [0, 999] in the answer portion
        integers = re.findall(r"\b(\d{1,3})\b", answer_text)
        for candidate in reversed(integers):
            val = int(candidate)
            return val
        return None


## Scaling Experiments Helpers

For sequential scaling (token-by-token generation) --> need generation that:
- monitors token stream incrementally
- detects thought boundaries
- intervenes dynamically

For WAIT intervention:
- detect end-think token before budget reached
- override generation
- continue chain

For parallel scaling:
- requires stochastic sampling
- multiple completions
- currently only evaluate single completion --> need answer = [multiple samples] then aggregation logic

Plots needed:
- accuracy vs thinking tokens
- total tokens vs accuracy
- error bars
- exact vs flexible curves

In [ ]:
budgets = [1024, 2048, 4000, 8000, 16000, 32000]
m_values = [1,2,4,8]

thinking_text = extract_thinking(model_output)
thinking_tokens = tokenizer.encode(thinking_text)

while not done:
    next_token = sample()

    if inside_thinking:
        thinking_token_count += 1

    if thinking_token_count > budget:
        inject_end_think_tag()

## Inference

You can also explore using vLLM to speed up inference!

In [ ]:
ENABLE_THINKING = True  # Set False for no-thinking condition
ANSWER_MODE = "exact_match"

model.to(device)
model.eval()

records = []

for i, example in enumerate(dataset):
    problem = example["problem"]
    gold_answer = int(example["answer"])

    messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."}, 
                {"role": "user", "content": problem}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    # Decode only the newly generated tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    model_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

    extracted = extract_answer(model_output, mode=ANSWER_MODE)
    if extracted is not None:
        correct = extracted == gold_answer
    else:
        correct = False

    records.append({
        "problem": problem,
        "gold_answer": gold_answer,
        "model_output": model_output,
        "extracted_answer": extracted,
        "correct": correct,
    })

    print(f"[{i+1}/{len(dataset)}] gold={gold_answer} pred={extracted} correct={correct}")

results_df = pd.DataFrame(records)
results_df

In [ ]:
results_df["correct"].mean()